In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap

plt.rcParams.update({"font.size": 12})

### Degree Distribution Feature Comparison

In [ ]:

# Load degree distribution percentile data
summary_path = "../inputs/degree_stats.csv"
df = pd.read_csv(summary_path)

std_col = "param_0"
alpha_col = "param_1"

df[alpha_col] = df[alpha_col].astype(float)
df[std_col] = df[std_col].astype(float)

In [ ]:
# Labels and values used in this plot
plot_features = ["mean", "p25", "median", "p75"]
plot_labels = ["Mean", "25th pct.", "Median", "75th pct."]
empirical_values = [126.1, 52, 97, 185,]
markers = ["X", "d", "^", "s", "o"]
std_values = np.sort(df[std_col].unique())
marker_by_std = {std: marker for std, marker in zip(std_values, markers)}

tick_exponents = np.arange(0, -11, -1)
alpha_ticks = 2.0 ** tick_exponents

In [ ]:
# Color scheme
sigma_colors = plt.cm.turbo(
    np.linspace(0.15, 0.85, len(std_values))
)
cmap = ListedColormap(sigma_colors)
color_by_std = dict(zip(std_values, sigma_colors))

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(7, 5), sharex=True, constrained_layout=True)
fig.supylabel("Degree", x=-0.04, y=0.55, fontsize=12)

for (
    ax,
    feature,
    y_label,
    empirical_value,
) in zip(
    axes,
    plot_features,
    plot_labels,
    empirical_values,
):
    # Subplots
    for std in std_values:
        sub = (df[df[std_col] == std].sort_values(alpha_col))
        alphas = sub[alpha_col].to_numpy(dtype=float)
        values = sub[feature].to_numpy(dtype=float)

        color = color_by_std[std]
        marker = marker_by_std[std]

        ax.plot(
            alphas,
            values,
            color=color,
            linewidth=1.5,
            marker=marker,
            markersize=6.0,
            markeredgecolor="black",
            markeredgewidth=0.5,
            zorder=3
        )

    # Empirical value horizontal lines
    ax.axhline(
        empirical_value,
        color="black",
        linestyle="--",
        linewidth=1.0,
        alpha=0.8,
        zorder=1,
    )
    ax.annotate(
        f"Empirical data: {empirical_value:g}",
        xy=(1, empirical_value),
        xytext=(-4, 4),
        textcoords="offset points",
        ha="right",
        va="bottom",
        fontsize=12,
    )

    ax.set_ylabel(y_label)


# Shared base-2 logarithmic x-axis
axes[-1].set_xscale("log", base=2)
axes[-1].set_xlim(2**-10 / 1.15, 1 * 1.15)
axes[-1].set_xticks(alpha_ticks)
axes[-1].set_xticklabels([
    rf"$2^{{{exponent}}}$"
    for exponent in tick_exponents
])
axes[-1].set_xlabel(r"Spatial freedom ($\alpha$)")

# Hide duplicate x tick labels on all but the bottom subplot.
for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)

# One shared legend for all four subplots
legend_handles = []
for std in std_values:
    legend_handles.append(
        Line2D(
            [0],
            [0],
            color=color_by_std[std],
            linewidth=1.2,
            marker=marker_by_std[std],
            markersize=6.0,
            markeredgecolor="black",
            markeredgewidth=0.5,
            label=rf"$\sigma = {std:g}$",
        )
    )

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.52, 1.0),
    ncol=len(std_values),
    frameon=False,
    columnspacing=1.5,
    handletextpad=0.5,
)

# Save figure
output_path = Path("../outputs/plots/fitting_degree_statistics_vs_alpha_by_std.png")
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, dpi=300, bbox_inches="tight")

plt.show()
plt.close(fig)

### Generating MSE Summary Tables

In [ ]:
degree_columns = ["mean", "p25", "median", "p75",]
empirical_values = np.array([
    126.1,  # mean
    52.0,   # 25th percentile
    97.0,   # median
    185.0,  # 75th percentile
])

In [ ]:
table_df = df.copy()
alpha_values = table_df[alpha_col].to_numpy(dtype=float)

with np.errstate(divide="ignore", invalid="ignore"):
    exponent_values = np.log2(alpha_values)

rounded_exponents = np.rint(exponent_values)

# Keep only integer exponents
valid_alpha = (
    np.isfinite(exponent_values)
    & (alpha_values > 0)
    & np.isclose(
        exponent_values,
        rounded_exponents,
        atol=1e-8,
        rtol=0,
    ))

table_df = table_df.loc[valid_alpha].copy()
table_df["alpha_exponent"] = (
    rounded_exponents[valid_alpha].astype(int)
)

In [ ]:
# Calculate overall MSE
empirical_values = np.array([
    126.1,  # mean
    52.0,   # 25th percentile
    97.0,   # median
    185.0,  # 75th percentile
])

table_df["mse"] = [
    mean_squared_error(
        empirical_values,
        row[degree_columns].to_numpy(dtype=float),
    )
    for _, row in table_df.iterrows()
]

In [ ]:
# Group sigma values and list alpha from 1 down to 2^-10.
table_df = table_df.sort_values(
    [std_col, "alpha_exponent"],
    ascending=[True, True],
).reset_index(drop=True)

In [ ]:
# Construct LaTeX body
body_lines = []
grouped = table_df.groupby(
    std_col,
    sort=True,
    dropna=False,
)

for sigma, sigma_group in grouped:
    sigma_group = sigma_group.reset_index(drop=True)
    n_rows = len(sigma_group)

    for row_index, row in sigma_group.iterrows():
        exponent = int(row["alpha_exponent"])

        alpha_latex = rf"$2^{{{exponent}}}$"

        if row_index == 0:
            sigma_entry = (
                rf"\multirow{{{n_rows}}}{{*}}"
                rf"{{$ {sigma:g} $}}"
            )
        else:
            sigma_entry = ""

        mean_text = f"{row['mean']:,.1f}"
        p25_text = f"{row['p25']:,.1f}"
        median_text = f"{row['median']:,.1f}"
        p75_text = f"{row['p75']:,.1f}"
        mse_text = f"{row['mse']:,.2f}"

        body_lines.append(
            f"{sigma_entry} & "
            f"{alpha_latex} & "
            f"{mean_text} & "
            f"{p25_text} & "
            f"{median_text} & "
            f"{p75_text} & "
            f"{mse_text} \\\\"
        )

    body_lines.append(r"\midrule")

# Remove final separator
if body_lines and body_lines[-1] == r"\midrule":
    body_lines.pop()

body = "\n".join(body_lines)

In [ ]:
# Construct complete LaTeX table
latex_table = rf"""
\begin{{table}}[H]
    \centering
    \footnotesize
    \caption{{}}
    \label{{tab:degree-statistics-fitting}}
    \begin{{tabular}}{{ccrrrrr}}
        \toprule
        Standard deviation ($\sigma$) &
        Spatial freedom ($\alpha$) &
        Mean &
        25th &
        Median &
        75th &
        MSE \\
        \midrule
{body}
        \bottomrule
    \end{{tabular}}
\end{{table}}
""".strip()

In [ ]:
print(latex_table)

In [ ]:
# Save
output_path = Path("../outputs/tex/degree_statistics_mse_table.tex")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(latex_table, encoding="utf-8")